# 第3章 ハンズオン：Tool呼び出しとMemory (Checkpointer) の実装

## 学習目標
- `@tool` デコレータでカスタムツールを定義する
- `create_agent` にツールを組み込む
- `InMemorySaver`（Checkpointer）で会話の記憶を実装する
- `response_format` で構造化出力（Structured Output）を使う

---

In [ ]:
%pip install -qU langchain langchain-openai langgraph pydantic

In [ ]:
import os
from google.colab import userdata

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = "chap03-handson"
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("環境変数の設定が完了しました")

## Part 1: カスタムツールの定義

`@tool` デコレータを使って、LLM が呼び出せるカスタムツールを定義します。

**重要なポイント:**
- 関数名 → ツール名になる
- docstring → LLM がいつ使うかを判断するための説明文になる
- 型ヒント → LLM が引数を正しく渡すために必要

In [ ]:
from langchain.tools import tool

# ツール1: 都市の天気を返すダミーツール
# 実際の天気APIの代わりに固定値を返す（学習用）
@tool
def get_weather(city: str) -> str:
    """指定した都市の現在の天気情報を取得します。
    
    Args:
        city: 天気を調べたい都市名（例: 東京、大阪、名古屋）
    """
    # 学習用のダミーデータ（実際のAPIの代わり）
    weather_data = {
        "東京": "晴れ、気温20°C、湿度50%",
        "大阪": "曇り、気温18°C、湿度60%",
        "名古屋": "雨、気温15°C、湿度80%",
    }
    return weather_data.get(city, f"{city}の天気情報は取得できませんでした")


# ツール2: 四則演算ツール
@tool
def calculate(expression: str) -> str:
    """数学の計算式を計算して結果を返します。
    
    Args:
        expression: 計算したい数式（例: '100 + 200 * 3'、'(50 - 10) / 4'）
    """
    try:
        # 安全な eval（数値と基本演算子のみ許可）
        result = eval(expression, {"__builtins__": {}}, {})
        return f"計算結果: {expression} = {result}"
    except Exception as e:
        return f"計算エラー: {str(e)}"


# ツールの動作確認
print("ツールのテスト:")
print(get_weather.invoke({"city": "東京"}))
print(calculate.invoke({"expression": "(100 + 200) * 3"}))

## Part 2: ツール付きエージェントの作成

In [ ]:
from langchain.agents import create_agent

# ツールをリストにして create_agent に渡す
agent_with_tools = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, calculate],   # 定義したツールを追加
    system_prompt=(
        "あなたは親切なAIアシスタントです。\n"
        "天気の質問には get_weather ツールを、\n"
        "計算が必要な場合は calculate ツールを使ってください。\n"
        "日本語で回答してください。"
    ),
)

# ツールが使われることを確認
result = agent_with_tools.invoke({
    "messages": [
        {"role": "user", "content": "東京の天気を教えてください。また、123 × 456 はいくつですか？"}
    ]
})

print("=== エージェントの回答 ===")
print(result["messages"][-1].content)
print()
print("=== 実行されたメッセージの流れ ===")
for msg in result["messages"]:
    print(f"  [{type(msg).__name__}]", end=" ")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        for tc in msg.tool_calls:
            print(f"→ ツール呼び出し: {tc['name']}({tc['args']})", end=" ")
    print()

## Part 3: Checkpointer（Memory）の追加

`InMemorySaver` を使うと、会話履歴が `thread_id` ごとに保存されます。

**thread_id とは？**
- 会話セッションを識別する文字列（ユーザーIDや会話IDなど）
- 同じ `thread_id` を使い続けることで、会話の記憶が継続する

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver: プロセスメモリに会話を保存するシンプルな実装
# （本番環境では RedisSaver / SqliteSaver などを使用する）
memory = InMemorySaver()

# Checkpointer を組み込んだエージェント
agent_with_memory = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather, calculate],
    system_prompt=(
        "あなたは親切なAIアシスタントです。"
        "日本語で回答し、必要に応じてツールを使ってください。"
    ),
    checkpointer=memory,   # ← ここが重要！Checkpointer を追加する
)

# thread_id で会話セッションを管理する
config = {"configurable": {"thread_id": "user_001"}}

# --- ターン1 ---
result1 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "東京の天気を教えてください"}]},
    config=config,   # config を渡すことで thread_id を指定する
)
print("[ターン1の回答]")
print(result1["messages"][-1].content)
print()

In [ ]:
# --- ターン2（前の会話を踏まえた質問）---
# 「調べた都市」を明示せずに聞いても、前の会話から文脈を理解できるか確認
result2 = agent_with_memory.invoke(
    {"messages": [{"role": "user", "content": "さっき調べた都市の気温は何度でしたか？"}]},
    config=config,   # 同じ thread_id を使って会話を継続する
)
print("[ターン2の回答（記憶を確認）]")
print(result2["messages"][-1].content)

## Part 4: Structured Output（構造化出力）

`response_format` に Pydantic モデルを指定すると、エージェントの出力を
決まった JSON 形式で取得できます。RAG システムや API レスポンスに便利です。

In [ ]:
from pydantic import BaseModel, Field

# 出力スキーマを Pydantic で定義
class WeatherReport(BaseModel):
    """天気レポートの構造化データ"""
    city: str = Field(description="都市名")
    condition: str = Field(description="天気の状態（晴れ/曇り/雨など）")
    temperature_celsius: float = Field(description="気温（摂氏）")
    recommendation: str = Field(description="天気に基づく行動アドバイス")


# Structured Output エージェント
structured_agent = create_agent(
    model="openai:gpt-4o",
    tools=[get_weather],
    response_format=WeatherReport,   # ← 出力スキーマを指定
    system_prompt=(
        "天気ツールを使って情報を取得し、指定のフォーマットで回答してください。"
    ),
)



In [ ]:
result = structured_agent.invoke({
    "messages": [{"role": "user", "content": "大阪の天気を調べて、外出するべきか教えてください"}]
})

# structured_response: response_format で指定したモデルのインスタンスが返る
report = result.get("structured_response")
if report:
    print("=== 構造化された天気レポート ===")
    print(f"都市: {report.city}")
    print(f"天気: {report.condition}")
    print(f"気温: {report.temperature_celsius}°C")
    print(f"おすすめ: {report.recommendation}")

## 【オプション】対話的な実行

`input()` 関数を使って、Google Colab 上で直接プロンプトを入力してエージェントを実行してみましょう。

In [ ]:
# Pythonの input() を使って対話的に質問を入力します
# 実行すると入力欄が表示されるので、質問を入力してEnterを押してください
interactive_input = input("エージェントへの入力: ")

In [ ]:
# スレッドIDを変えて実行します（新しい会話として開始）
interactive_config = {"configurable": {"thread_id": "interactive_thread"}}


In [ ]:
from langchain_core.messages import HumanMessage

response_interactive = agent.invoke(
    {"messages": [HumanMessage(content=interactive_input)]},
    config=interactive_config
)
print("回答:")
print(response_interactive["messages"][-1].content)

---

## まとめ

| 機能 | 使用したもの | ポイント |
|------|------------|--------|
| カスタムツール | `@tool` デコレータ | docstring と型ヒントが LLM のツール選択精度を左右する |
| 会話の記憶 | `InMemorySaver` + `thread_id` | `config` で thread_id を渡すことで会話が継続する |
| 構造化出力 | `response_format=PydanticModel` | `result["structured_response"]` でモデルインスタンスを取得 |

**次の章（第4章）では、** 外部サービスとの連携（MCP）と、
重要な操作の前に人間の確認を求める HITL を実装します。